# Comprehensive Training Pipeline — Surface Crack Detection

**Full-spectrum training: Standard → K-Fold CV → Knowledge Distillation → Synthetic Augmentation**

This notebook orchestrates the entire ML workflow:

| # | Phase | Est. Time | What |
|---|-------|-----------|------|
| 1 | Setup + Data | ~1 min | Clone, deps, dataset prep |
| 2 | Standard Training | ~15 min | ResNet50, EfficientNet-B0, ViT-B/16 |
| 3 | 5-Fold Cross-Validation | ~10 min | Stratified K-fold on ResNet50 |
| 4 | Ensemble Eval | ~30 sec | Softmax averaging across 3 models |
| 5 | Knowledge Distillation | ~5 min | MobileNetV3-Small from teacher ensemble |
| 6 | Synthetic Augmentation | ~10 min | SD-generated images + retrain |
| 7 | Reports + Push | ~1 min | Session save, viz, optional HF push |

**Runtime:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU**

In [ ]:
# Cell 1: Clone repo, install deps, path setup
import sys, os, subprocess, json, time

PROJECT_DIR = "/content/bootcamp-ace-26-team-7"
GIT_URL = "https://github.com/esetu-git-public/bootcamp-ace-26-team-7.git"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {GIT_URL} ...")
    result = subprocess.run(
        ["git", "clone", GIT_URL, PROJECT_DIR],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        print(f"Clone failed:\n{result.stderr}")
        raise SystemExit(1)
    print("Clone complete.")
else:
    print(f"{PROJECT_DIR} already exists — pulling latest")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], capture_output=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")

# Install dependencies (CUDA version for GPU training)
print("Installing dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
     "--extra-index-url", "https://download.pytorch.org/whl/cu124"],
    capture_output=True, text=True, timeout=300
)
if result.returncode != 0:
    print(f"pip warnings/errors:\n{result.stderr[-500:]}")
print("Dependencies installed.")

In [ ]:
# Cell 2: Hardware detection and device selection
from src.monitor import detect_hardware, auto_monitor
import torch
import src.config as cfg

detect_hardware(verbose=True)

devices = {}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    devices["cuda"] = f"GPU {torch.cuda.get_device_name(0)} ({props.total_memory / 1024**3:.1f} GB)"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    devices["mps"] = "Apple MPS (Metal)"
devices["cpu"] = "CPU (slow — use only for testing)"

keys = list(devices.keys())
print("\nDetected compute devices:")
for i, k in enumerate(keys):
    print(f"  [{i}] {k.upper()}: {devices[k]}")

non_cpu = [k for k in keys if k != "cpu"]
default_idx = 0 if non_cpu else keys.index("cpu")
try:
    choice = input(f"Select device [0-{len(keys)-1}], Enter=auto ({keys[default_idx].upper()}): ").strip()
    selected = keys[int(choice)] if choice else keys[default_idx]
except (ValueError, IndexError):
    print(f"Invalid choice, using default: {keys[default_idx].upper()}")
    selected = keys[default_idx]

cfg.Config.DEVICE = torch.device(selected)
print(f"\nUsing: {selected.upper()} — {devices[selected]}")

auto_monitor()

In [ ]:
# Cell 3: Download dataset + prepare splits (standard + k-fold)
from huggingface_hub import hf_hub_download
import zipfile

REPO_ID = "amruthjakku/surface-crack-detection-model"
RAW_DIR = "data/raw"

if os.path.exists(RAW_DIR) and os.listdir(RAW_DIR):
    print(f"Dataset found at {RAW_DIR}/ — skipping download")
else:
    try:
        print("Downloading dataset from HF Hub...")
        os.makedirs("data", exist_ok=True)
        zip_path = hf_hub_download(repo_id=REPO_ID, filename="dataset.zip", repo_type="model")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall("data")
        print(f"Extracted to {RAW_DIR}/")
    except Exception as e:
        print(f"Dataset download failed: {e}")
        print("Place images manually in data/raw/{Cracks,Patch,Potholes,Surface Defects}/")
        raise SystemExit(1)

# Standard 70/15/15 split
from src.prepare_data import prepare_data, generate_kfold_splits

print("\n--- Standard Split ---")
prepare_data()

# K-Fold splits
print("\n--- K-Fold Splits ---")
generate_kfold_splits()

# Verify
from src.dataset import get_dataloaders
train_loader, val_loader, test_loader = get_dataloaders()
print(f"\nStandard split: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}")

if len(train_loader.dataset) == 0:
    print("ERROR: Training set empty.")
    raise SystemExit(1)

# Show class distribution
print("\nClass distribution (raw):")
for class_name in cfg.Config.CLASSES:
    class_dir = os.path.join(RAW_DIR, class_name)
    if os.path.isdir(class_dir):
        count = len([f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        print(f"  {class_name}: {count}")

In [ ]:
# Cell 4: Standard Training — ResNet50, EfficientNet-B0, ViT-B/16
from src.train import run_training
from src.evaluate import run_evaluation
from src.session import SessionTracker

MODELS = ["resnet50", "efficientnet_b0", "vit_b_16"]
model_results = {}

for model_name in MODELS:
    print(f"\n{'=' * 60}")
    print(f"Training {model_name}...")
    print(f"{'=' * 60}")
    t0 = time.time()
    try:
        train_result = run_training(model_name=model_name)
        elapsed = time.time() - t0
        print(f"\nTraining time: {elapsed / 60:.1f} min")

        path = cfg.Config.get_model_path(model_name)
        if os.path.exists(path):
            print(f"Checkpoint: {path} ({os.path.getsize(path)/1024/1024:.1f} MB)")
            metrics = run_evaluation(model_name=model_name)
            if metrics:
                model_results[model_name] = {"status": "ok", **metrics}
            else:
                model_results[model_name] = {"status": "ok"}
        else:
            print("No checkpoint saved.")
            model_results[model_name] = {"status": "failed", "error": "No checkpoint"}
    except Exception as e:
        print(f"{model_name} training failed: {e}")
        model_results[model_name] = {"status": "failed", "error": str(e)[:200]}
    torch.cuda.empty_cache()

print("\nStandard training complete.")
print("Model results so far:")
for name, res in model_results.items():
    acc = res.get("accuracy", "?")
    f1 = res.get("weighted_f1", "?")
    print(f"  {name:20s} status={res.get('status','?')} acc={acc} f1={f1}")

In [ ]:
# Cell 5: 5-Fold Cross-Validation
print(f"{'=' * 60}")
print("5-FOLD CROSS-VALIDATION")
print(f"{'=' * 60}")

from src.train import run_kfold

kfold_result = run_kfold(model_name="resnet50")

print("\nK-Fold CV complete.")

In [ ]:
# Cell 6: Ensemble Evaluation — softmax averaging across all trained models
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from src.model import get_model
from src.dataset import get_dataloaders

device = cfg.Config.DEVICE
available_models = [m for m in MODELS if os.path.exists(cfg.Config.get_model_path(m))]

if len(available_models) < 2:
    print(f"Ensemble skipped — need >= 2 trained models, have {len(available_models)}")
    ensemble_metrics = {}
else:
    nets = []
    for name in available_models:
        net = get_model(model_name=name, num_classes=cfg.Config.NUM_CLASSES, pretrained=False)
        net.load_state_dict(torch.load(cfg.Config.get_model_path(name), map_location=device))
        nets.append(net.to(device).eval())
        print(f"Loaded {name}")

    _, _, test_loader = get_dataloaders()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = sum(torch.softmax(net(images), dim=1) for net in nets) / len(nets)
            y_true.extend(labels.numpy())
            y_pred.extend(torch.max(probs, 1)[1].cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=cfg.Config.CLASSES, zero_division=0)

    print(f"\n{'=' * 60}")
    print(f"ENSEMBLE ({' + '.join(available_models)})")
    print(f"{'=' * 60}")
    print(f"Test Accuracy:   {acc:.4f}")
    print(f"Weighted F1:     {f1_w:.4f}")
    print(f"\nClassification Report:\n{report}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cfg.Config.CLASSES, yticklabels=cfg.Config.CLASSES)
    plt.title(f"Ensemble Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    cm_path = os.path.join(cfg.Config.REPORTS_DIR, "confusion_matrix_ensemble.png")
    os.makedirs(cfg.Config.REPORTS_DIR, exist_ok=True)
    plt.savefig(cm_path)
    plt.show()
    print(f"Saved: {cm_path}")

    ensemble_metrics = {"accuracy": round(acc, 4), "weighted_f1": round(f1_w, 4)}

In [ ]:
# Cell 7: Knowledge Distillation — MobileNetV3-Small from Teacher Ensemble
print(f"{'=' * 60}")
print("KNOWLEDGE DISTILLATION")
print(f"{'=' * 60}")
print(f"Student: {cfg.Config.DISTILL_STUDENT}")
print(f"Teachers: {cfg.Config.DISTILL_TEACHERS}")
print(f"Temperature: {cfg.Config.DISTILL_TEMPERATURE}, Alpha: {cfg.Config.DISTILL_ALPHA}")

from src.distill import run_distillation

student_metrics = run_distillation()

student_path = os.path.join(cfg.Config.MODELS_DIR, f"student_{cfg.Config.DISTILL_STUDENT}.pth")
if student_metrics and os.path.exists(student_path):
    size_mb = os.path.getsize(student_path) / (1024 * 1024)
    print(f"\nStudent checkpoint: {student_path} ({size_mb:.1f} MB)")
    print(f"Best val loss: {student_metrics['best_val_loss']:.4f}")
    print(f"Best val acc: {student_metrics['best_val_acc']:.4f}")
else:
    print("Distillation did not produce a checkpoint.")

print("\nDistillation complete.")

In [ ]:
# Cell 8: Synthetic Augmentation via Stable Diffusion
print(f"{'=' * 60}")
print("SYNTHETIC AUGMENTATION")
print(f"{'=' * 60}")
print("This step generates 100 synthetic images per class using Stable Diffusion v1.5.")
print("Requires: diffusers, transformers, GPU with 8+ GB VRAM.")

synth_input = input("\nRun synthetic augmentation? (y/N): ").strip().lower()

if synth_input in ("y", "yes"):
    from src.generate_synthetic import generate_synthetic
    try:
        cfg.Config.SYNTHETIC_ENABLED = True
        generate_synthetic()

        # Quick retrain with synthetic data enabled
        print("\n--- Retraining with augmented data ---")
        from src.train import run_training
        orig_epochs = (cfg.Config.EPOCHS_WARMUP, cfg.Config.EPOCHS_FINE_TUNE)
        cfg.Config.EPOCHS_WARMUP = 2
        cfg.Config.EPOCHS_FINE_TUNE = 5

        syn_metrics = run_training(model_name="resnet50")
        print(f"Augmented training complete. Best val loss: {syn_metrics['best_val_loss']:.4f}")

        cfg.Config.EPOCHS_WARMUP, cfg.Config.EPOCHS_FINE_TUNE = orig_epochs
        print("Synthetic augmentation complete.")
    except Exception as e:
        print(f"Synthetic augmentation failed: {e}")
        cfg.Config.SYNTHETIC_ENABLED = False
else:
    print("Skipped synthetic augmentation.")

In [ ]:
# Cell 9: Save session to history
all_models = {**model_results}
if ensemble_metrics:
    all_models["ensemble"] = {"status": "ok", "models_used": available_models, **ensemble_metrics}
if "student_metrics" in dir() and student_metrics:
    all_models["student_mobilenet_v3_small"] = {"status": "ok", **student_metrics}

kfold_summary = kfold_result if "kfold_result" in dir() and kfold_result else None

entry = SessionTracker.new_session(
    device=cfg.Config.DEVICE,
    mode="comprehensive",
    model_results=all_models,
    kfold_summary=kfold_summary,
)
print(f"Session saved: {entry['timestamp']}")

In [ ]:
# Cell 10: Summary with comparison chart
from src.visualize import plot_session_comparison
from IPython.display import display, Image

print("=" * 60)
print("COMPREHENSIVE TRAINING SUMMARY")
print("=" * 60)
print(f"Device: {cfg.Config.DEVICE}")
print(f"Mode: Comprehensive (standard + kfold + distillation + synthetic)")
print()

for name in MODELS:
    info = model_results.get(name, {})
    status = info.get("status", "?")
    if status == "ok":
        acc = info.get("accuracy", "?")
        f1 = info.get("weighted_f1", "?")
        print(f"  {name:20s} Acc={acc}  F1={f1}")
    else:
        print(f"  {name:20s} {status}")

if ensemble_metrics:
    print(f"  {'Ensemble':20s} Acc={ensemble_metrics['accuracy']}  F1={ensemble_metrics['weighted_f1']}")

if "kfold_result" in dir() and kfold_result:
    print(f"  {'K-Fold CV':20s} Acc={kfold_result['avg_val_acc']:.4f} ± {kfold_result['std_val_acc']:.4f}")

if "student_metrics" in dir() and student_metrics:
    print(f"  {'Distilled Student':20s} Val Acc={student_metrics['best_val_acc']:.4f}")

print(f"\n{SessionTracker.summary()}")

viz_path = plot_session_comparison()
if viz_path:
    display(Image(viz_path))
else:
    print("Not enough sessions to plot comparison.")

In [ ]:
# Cell 11: Push trained models and reports to HuggingFace Hub (optional)
HF_TOKEN = input("Enter HF token (write access) or press Enter to skip: ").strip()

if not HF_TOKEN:
    print("Skipped push.")
else:
    from huggingface_hub import HfApi, login
    try:
        login(token=HF_TOKEN)
        api = HfApi()
        pushed = 0

        for model_name in MODELS:
            path = cfg.Config.get_model_path(model_name)
            if not os.path.exists(path):
                continue
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=f"{model_name}_best.pth",
                repo_id=REPO_ID, repo_type="model",
            )
            print(f"Pushed {model_name}_best.pth")
            pushed += 1

        # Push student model if exists
        student_path = os.path.join(cfg.Config.MODELS_DIR, f"student_{cfg.Config.DISTILL_STUDENT}.pth")
        if os.path.exists(student_path):
            api.upload_file(
                path_or_fileobj=student_path,
                path_in_repo=f"student_{cfg.Config.DISTILL_STUDENT}.pth",
                repo_id=REPO_ID, repo_type="model",
            )
            print(f"Pushed student_{cfg.Config.DISTILL_STUDENT}.pth")
            pushed += 1

        # Push reports
        if os.path.isdir(cfg.Config.REPORTS_DIR):
            for fname in os.listdir(cfg.Config.REPORTS_DIR):
                fpath = os.path.join(cfg.Config.REPORTS_DIR, fname)
                if not os.path.isfile(fpath):
                    continue
                api.upload_file(
                    path_or_fileobj=fpath,
                    path_in_repo=f"reports/{fname}",
                    repo_id=REPO_ID, repo_type="model",
                )
                print(f"Pushed reports/{fname}")

        print(f"\nPushed {pushed} model(s) to {REPO_ID}")
    except Exception as e:
        print(f"Push failed: {e}")

In [ ]:
print("=" * 60)
print("COMPREHENSIVE TRAINING COMPLETE")
print("=" * 60)
print("Models: models/")
print("Reports: reports/")
print("Session: reports/session_results.json")
print()
print("What was done:")
print("  1. Standard training — 3 models (ResNet50, EfficientNet-B0, ViT-B/16)")
print("  2. 5-Fold Cross-Validation — stratified, with aggregate metrics")
print("  3. Ensemble evaluation — softmax averaging across models")
print("  4. Knowledge Distillation — MobileNetV3-Small from teacher ensemble")
print("  5. Synthetic Augmentation — Stable Diffusion generated images + retrain")
print()
print("Next steps:")
print("  - Deploy the student model for lightweight inference (models/student_*.pth)")
print("  - Retrain with Config.SYNTHETIC_ENABLED=True for full augmented training")